# Azure OpenAI RAG with Vector Store (Responses API)

This notebook demonstrates a simple retrieval-augmented generation (RAG) flow using:

- An existing **Azure OpenAI Vector Store** (already deployed)
- Your deployed **Azure OpenAI model deployment** (used by Responses API)

No Azure AI Search is used.

You will: (1) configure connection variables, (2) create a client, (3) call `responses.create` with `file_search` tool resources, and (4) print the answer.

## 1) Configuration (fill in your values)

Set these as environment variables (recommended) or edit the defaults below.

Required:
- `AZURE_OPENAI_API_KEY`
- `AZURE_OPENAI_ENDPOINT`
- `AZURE_OPENAI_API_VERSION` (if not set, we default to `2025-03-01-preview`)
- `AZURE_OPENAI_DEPLOYMENT` (your deployed Azure OpenAI model name/deployment)
- `AZURE_OPENAI_VECTOR_STORE_ID` (the Vector Store you already deployed)

Optional:
- `AZURE_OPENAI_ASSISTANT_NAME` (only used for display)

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)

AZURE_OPENAI_API_KEY = os.getenv('AZURE_OPENAI_API_KEY')
AZURE_OPENAI_ENDPOINT = os.getenv('AZURE_OPENAI_ENDPOINT')
AZURE_OPENAI_API_VERSION = os.getenv('AZURE_OPENAI_API_VERSION', '2025-03-01-preview')

AZURE_OPENAI_DEPLOYMENT = os.getenv('AZURE_OPENAI_DEPLOYMENT')
AZURE_OPENAI_VECTOR_STORE_ID = os.getenv('AZURE_OPENAI_VECTOR_STORE_ID')

missing = [
    name
    for name, value in [
        ('AZURE_OPENAI_API_KEY', AZURE_OPENAI_API_KEY),
        ('AZURE_OPENAI_ENDPOINT', AZURE_OPENAI_ENDPOINT),
        ('AZURE_OPENAI_DEPLOYMENT', AZURE_OPENAI_DEPLOYMENT),
        ('AZURE_OPENAI_VECTOR_STORE_ID', AZURE_OPENAI_VECTOR_STORE_ID),
    ]
    if not value
]

if missing:
    raise ValueError('Missing required env vars: ' + ', '.join(missing))

print('Config OK')
print('API version:', AZURE_OPENAI_API_VERSION)
print('Model deployment:', AZURE_OPENAI_DEPLOYMENT)
print('Vector store:', AZURE_OPENAI_VECTOR_STORE_ID)


Config OK
API version: 2025-03-01-preview
Model deployment: gpt-5.4-nano
Vector store: vs_qpxO0nZ29dXjA0NwiaFn42rk


## 2) Create an Azure OpenAI client

In [5]:
from openai import AsyncAzureOpenAI

client = AsyncAzureOpenAI(
    api_key=AZURE_OPENAI_API_KEY,
    api_version=AZURE_OPENAI_API_VERSION,
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
)

print('Client created')

Client created


## 3) Responses API call with vector store retrieval

In [8]:
import asyncio

async def run_rag_responses(question: str):
    response = await client.responses.create(
        model=AZURE_OPENAI_DEPLOYMENT,
        input=question,
        tools=[
            {
                'type': 'file_search',
                'vector_store_ids': [AZURE_OPENAI_VECTOR_STORE_ID],
            }
        ],
    )

    output_parts = []
    for item in response.output:
        if getattr(item, 'type', None) == 'message':
            for content in getattr(item, 'content', []) or []:
                if getattr(content, 'type', None) == 'output_text':
                    output_parts.append(getattr(content, 'text', ''))
    return '\n'.join([p for p in output_parts if p]).strip()

question = 'What is GRPO?'
answer = await run_rag_responses(question)
print(answer)

GRPO stands for **Group Relative Policy Optimization**. It’s a **reinforcement learning (RL) fine-tuning** method used to improve **LLMs for math and reasoning tasks**. GRPO fine-tunes a model using **deterministic reward functions** (so you don’t need labeled “best answers”), by generating **multiple candidate responses** per prompt, assigning rewards to each, and using those aggregated rewards to update the model via a **GRPO loss**.


## 4) Try your own question

Edit `question` in the previous cell and re-run.